## PRT564 Assessment 4 — Model Evaluation
### Naive Bayes vs SVM vs Random Forest
**Group 15 — Sydney Campus**

In [ ]:
import os

BASE = r"C:\Users\ranas\OneDrive\Desktop\Rana_Research_Workspace\Rana_Research_Workspace"
CLF_OUT  = os.path.join(BASE, "Classification_Outputs")
EVAL_OUT = os.path.join(BASE, "Evaluation_Outputs")
os.makedirs(EVAL_OUT, exist_ok=True)

for p, n in [(CLF_OUT,"Classification_Outputs"),(EVAL_OUT,"Evaluation_Outputs")]:
    print(n, "found" if os.path.exists(p) else "NOT FOUND")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve, auc,
                              accuracy_score, precision_score, recall_score, f1_score)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.05)

In [ ]:
results_df = pd.read_csv(os.path.join(CLF_OUT, 'classification_results.csv'))
preds_df   = pd.read_csv(os.path.join(CLF_OUT, 'predictions.csv'))

y_test     = preds_df['y_true'].values
y_pred_nb  = preds_df['pred_NaiveBayes'].values
y_pred_svm = preds_df['pred_SVM'].values
y_pred_rf  = preds_df['pred_RandomForest'].values

y_proba_nb  = np.load(os.path.join(CLF_OUT, 'proba_NaiveBayes.npy'))
y_proba_svm = np.load(os.path.join(CLF_OUT, 'proba_SVM.npy'))
y_proba_rf  = np.load(os.path.join(CLF_OUT, 'proba_RandomForest.npy'))
classes = list(np.load(os.path.join(CLF_OUT, 'classes.npy'), allow_pickle=True))

print(f"test set size: {len(y_test)}  |  classes: {classes}")
print("\nresults:")
print(results_df.drop(columns=['Best_Params']).round(4).to_string(index=False))

## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
order = ['Decline','Normal','Boom']

for ax, (name, ypred) in zip(axes,
    [('Naive Bayes',   y_pred_nb),
     ('SVM (RBF)',     y_pred_svm),
     ('Random Forest', y_pred_rf)]):
    cm = confusion_matrix(y_test, ypred, labels=order)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=order, yticklabels=order, annot_kws={'size':14})
    acc = accuracy_score(y_test, ypred)
    f1  = f1_score(y_test, ypred, average='macro')
    ax.set_title(f'{name}\nAcc={acc:.3f}  F1={f1:.3f}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

fig.suptitle('A4 Fig E1 - Confusion Matrices', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUT, 'A4_figE1_confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

## ROC Curves (One-vs-Rest)

In [ ]:
y_test_bin = label_binarize(y_test, classes=classes)
n_classes = len(classes)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, yproba) in zip(axes,
    [('Naive Bayes',   y_proba_nb),
     ('SVM (RBF)',     y_proba_svm),
     ('Random Forest', y_proba_rf)]):
    for i, cls in enumerate(classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], yproba[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, lw=2, label=f'{cls} (AUC={roc_auc:.3f})')
    ax.plot([0,1],[0,1],'k--', lw=1.2, alpha=0.5)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{name}', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)

fig.suptitle('A4 Fig E2 - ROC Curves (One-vs-Rest)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUT, 'A4_figE2_roc_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## Model Comparison

In [ ]:
metrics = ['Test_Accuracy','Test_Precision','Test_Recall','Test_F1_macro','Test_F1_weighted','CV_F1_macro']
metric_labels = ['Accuracy','Precision','Recall','F1 (macro)','F1 (weighted)','CV F1 (macro)']

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(metric_labels))
w = 0.25
colors = ['#9B59B6','#3498DB','#27AE60']

for i, model in enumerate(results_df['Model']):
    vals = results_df.loc[results_df['Model']==model, metrics].values.flatten()
    bars = ax.bar(x + (i-1)*w, vals, w, label=model, color=colors[i], edgecolor='white', linewidth=1.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.005, f'{v:.3f}',
                ha='center', fontsize=8, fontweight='bold')

ax.set_xticks(x); ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('A4 Fig E3 - Model Performance Comparison', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUT, 'A4_figE3_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Feature Importance

In [ ]:
fi = pd.read_csv(os.path.join(CLF_OUT, 'feature_importance.csv'))
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=fi, x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title('A4 Fig E4 - Random Forest Feature Importance', fontweight='bold')
ax.set_xlabel('Importance Score'); ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig(os.path.join(EVAL_OUT, 'A4_figE4_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

## Per-Class Metrics

In [ ]:
rows = []
for name, ypred in [('Naive Bayes',y_pred_nb),('SVM',y_pred_svm),('Random Forest',y_pred_rf)]:
    for cls in classes:
        mask = y_test == cls
        if mask.sum() == 0: continue
        prec = precision_score(y_test, ypred, labels=[cls], average='macro', zero_division=0)
        rec  = recall_score(y_test, ypred, labels=[cls], average='macro', zero_division=0)
        f1   = f1_score(y_test, ypred, labels=[cls], average='macro', zero_division=0)
        rows.append({'Model':name,'Class':cls,'Precision':prec,'Recall':rec,'F1':f1,'Support':int(mask.sum())})

per_class = pd.DataFrame(rows)
print("Per-class metrics:")
print(per_class.round(4).to_string(index=False))
per_class.to_csv(os.path.join(EVAL_OUT, 'A4_per_class_metrics.csv'), index=False)

## Final Summary

In [ ]:
best_idx   = results_df['Test_F1_macro'].idxmax()
best_model = results_df.loc[best_idx, 'Model']
best_f1    = results_df.loc[best_idx, 'Test_F1_macro']

print("="*60)
print(f"RECOMMENDED MODEL: {best_model}")
print(f"Test F1 (macro): {best_f1:.4f}")
print("="*60)
print("\nfinal results:")
print(results_df.drop(columns=['Best_Params']).round(4).to_string(index=False))
results_df.to_csv(os.path.join(EVAL_OUT, 'A4_final_results.csv'), index=False)
print(f"\nall outputs saved to: {EVAL_OUT}")